# Project number 5 - Continued Fractions

In [2]:
# Here we will run all our imports 
import math
import time
import random
import numpy as np
from fractions import Fraction
from math import gcd, isqrt, log
from collections import Counter
import matplotlib.pyplot as plt

# Core 1 :
### In this Need to make two functions, One that computes the continued fraction, One that reconstructs a rational number from its continued fraction and then analyse and test the functions.
### From the question, For a given $θ ∈ R $,  define integers $a_j$ for $j≥0$ , we define $θ_0 = θ,$ $a_j =$ $\lfloorθ_j\rfloor$ 
### If $a_j = θ_j,$  stop ; 
### If $a_j ≠ θ_j,$  then define $θ_(j+1) > 1$ as $θ_j = a_j + 1/θ_(j+1)$. 
### We have :
### $
\theta = a_0 + \frac{1}{a_1 + \frac{1}{a_2 + \frac{1}{\ddots + \frac{1}{a_j + \cdots}}}} = [a_0; a_1, a_2, \dots, a_j, \dots]
$


In [5]:
# First function
def continued_fraction(theta, n):
    cf = []
    theta_i = theta

    for i in range(n+1):
        # Define a as required in question
        a = math.floor(theta_i)
        cf.append(a)

        remainder = theta_i - a

        # Stop the loop if its an exact integer (a = theta)
        if abs(remainder) < 1e-9:
            break

        theta_i = 1 / remainder

    return cf

# Second function
def reconstructed_rational(cf):
    if not cf:
        return 0

    # Begin fraction
    fraction = cf[-1] / 1

    # Work backwards through a
    for a in reversed(cf[:-1]):
        fraction = a + (1 / fraction)
        
    return fraction

# Now to test and analyse :

# Test with rational number
cf_rational = continued_fraction(4/5, n = 10)
print("4/5 expanded is :", cf_rational , "\n")

# Test with Irrational (root 5)
cf_irrational = continued_fraction(math.sqrt(5), n = 10)
print("Root 5 expanded is :", cf_irrational, "\n")
print("Root 5 reconstructed is :", reconstructed_rational(cf_irrational), "\n")

# Execution time with pythons in built speed test. Reseting timer (t0) to 0 each time.
t0 = time.perf_counter()
cf = continued_fraction(math.sqrt(5), n=10)
t_cf = (time.perf_counter() - t0) * 1e6

t0 = time.perf_counter()
frac = reconstructed_rational(cf)
t_rec = (time.perf_counter() - t0) * 1e6

print(f"Continued fraction time: {t_cf:.2f} microseconds", "\n")
print(f"Reconstructed rational time: {t_rec:.2f} microseconds")


4/5 expanded is : [0, 1, 4] 

Root 5 expanded is : [2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4] 

Root 5 reconstructed is : 2.236067977499718 

Continued fraction time: 82.73 microseconds 

Reconstructed rational time: 62.96 microseconds


### Through our analysis we find that the reconstructed function is faster. 

# Core 2 

# Core 3

In [4]:
def sample_fractions(Q, N):
    fractions = []

#1 on first line, 2 on next... , and the total is just sum of all the lines. (single slash gives float not integer as well.)
    total = Q * (Q - 1) // 2

    while len(fractions) < N:

        r = random.randrange(total)

#isqrt is sqrt and then rounded down#
#this line and for a as well from ai so change if needed- lwk spent way too much time on it and couldnt work it out-#
#turns list from 0, 12, 345, ... into denominator#
        q = (1 + isqrt(1 + 8*r)) // 2 + 1

        first = (q - 2) * (q - 1) // 2
        a = r - first + 1

        if gcd(a, q) == 1:
            fractions.append((a, q))
    return fractions


def cf_length(a, q):
#this is for the s1(x)#
    length = 0

#Euclidean algorithm without needing to use part 1 (using question 4)#
    while a != 0:
        a, q = q % a, a      # a/q -> q/a (->) (qmod(a))/a#
        length += 1  
    return length


def calculate_S(fractions):

    S_one = []
    S_log = []

    for a, q in fractions:

# phi(x) = 1#
        S_one.append(cf_length(a, q))

# phi(x) = log(1/x)#
#all the bits end up cancelling out just leaving the denominator (logs added mean inside multiplied)#        
        S_log.append(log(q))

    return np.array(S_one), np.array(S_log)


Q_values = [10**2, 10**3, 10**4, 10**5, 10**6]

mean_one = []
variance_one = []
means_log = []
variances_log = []


for Q in Q_values:

    fractions = sample_fractions(Q, 50000)

    S_one, S_log = calculate_S(fractions)

#for phi=1
    mu_one = np.mean(S_one)
    var_one = np.mean((S_one - mu_one)**2)

    mean_one.append(mu_one)
    variance_one.append(var_one)

#for phi = log...#
    mu_log = np.mean(S_log)
    var_log = np.mean((S_log - mu_log)**2)

    means_log.append(mu_log)
    variances_log.append(var_log)


for i, Q in enumerate(Q_values):

    print("\nQ =", Q)

    print("phi = 1:")
    print("Mean =", mean_one[i])
    print("Variance =", variance_one[i])

    print("phi = log(1/x):")
    print("Mean =", means_log[i])
    print("Variance =", variances_log[i])


#might be useful to use every fraction if its for a low number instead of just a sample, so:
def all_fractions(Q):
    fractions = []

    for q in range(2, Q + 1):
        for a in range(1, q):
            if gcd(a, q) == 1:
                fractions.append((a, q))

     return fractions        


#This takes the form of the fibbonacci sequence. This is because: instead of looking for smallest k, look for the largest l for 1,2,3,4... . Here you find that the subsequent phases of the algorithm use smaller numbers that are previous.#
#This therefore means that to find the smallest number, the second line of working must be the smallest number for the number below and so on.#
#Taking the form of k=1*(k-1)+(k-2), where k is the term of the fibbonacci k-1 is the one before and k-2 is the one before that. It is multiplied by 1 as there is symmetry for both sides and thus using 2 or 3 wouldnt make a larger number of steps#
#This would make the next line (k-1)=1*(k-2)+(k-3)... . This is now the optimal result for the number below and as one line of working has been used up, that makes it the optimal result for that value of k.

#Therefore for the smallest denominator, take k.....


def fib(n):
    a, b = 1, 2          #values for n=0 and n=1#
    for p in range(n - 1):
        a, b = b, a + b
    return b

print(fib(6))

IndentationError: unindent does not match any outer indentation level (<string>, line 103)

# Core 4

In [ ]:
# Generate all reduced fractions a/q with 0 < a < q <= Q.
# Store fractions exactly as numerator-denominator pairs.
def core4_fractions(Q):
    if Q < 2:
        raise ValueError("Q must be at least 2.")

    for q in range(2, Q + 1):
        for a in range(1, q):
            if gcd(a, q) == 1:
                yield a, q


# Return [a1, ..., ak] for a/q = [0; a1, ..., ak].
def core4_coefficients(a, q):
    if not 0 < a < q:
        raise ValueError("Require 0 < a < q.")

    coefficients = []

    while a != 0:
        coefficient, remainder = divmod(q, a)
        coefficients.append(coefficient)
        q, a = a, remainder

    return coefficients

def core4_length(a, q):
    return len(core4_coefficients(a, q))


def core4_fibonacci(n):
    a, b = 0, 1

    for _ in range(n):
        a, b = b, a + b

    return a


# Check a few known examples.
assert core4_coefficients(1, 2) == [2]
assert core4_coefficients(2, 3) == [1, 2]
assert core4_coefficients(3, 5) == [1, 1, 2]
assert core4_coefficients(5, 8) == [1, 1, 1, 2]

print("The example checks passed.\n")

## Minimum denominator for a given length
# Search in increasing denominator order.
def core4_minimum_denominators(Q):
    minimums = {}

    for a, q in core4_fractions(Q):
        k = core4_length(a, q)

        if k not in minimums:
            minimums[k] = (a, q)

    return minimums


search_Q = 200
minimums = core4_minimum_denominators(search_Q)

print("Length | Example fraction | Minimum denominator | Fibonacci prediction")

for k in sorted(minimums):
    a, q = minimums[k]
    predicted = core4_fibonacci(k + 2)

    print(f"{k:6d} | {a:3d}/{q:<11d} | {q:19d} | {predicted:20d}")

    assert q == predicted
    assert core4_coefficients(a, q) == [1] * (k - 1) + [2]

# Extension - 7

In [ ]:
#making the function to compute the frequency
"""
to declutter what the notation is saying in words its asking for the cardinality of the set of ordered pairs of
(x,j) such that x is an array taken from the random sample of f_Q and j is an index in in the array of x such that
a_j  = k, where k is our input parameter and then we divide this expression by the sum of the lengths from all x in
the random sample
"""

#this function uses logic defined in the previous sections such as:
#the continued fraction function is defined in core 1
#the random sample and f_Q logic is defined in core 3
#ill copy them over as a comment just so that you dont have to scroll back up

#exact version of continued fractions (always terminates for rational theta)
def continued_fraction(theta):
    cf = []

    while theta != 0:
        a = math.floor(theta)
        cf.append(a)

        remainder = theta - a

        if remainder == 0:
            break

        theta = 1 / remainder

    return cf

def f_Q(Q):
    f_Q = []
    for q in range(2,Q+1):
        for a in range(1,q):
            if math.gcd(a,q) == 1:
                f_Q.append(Fraction(a,q))
    return f_Q 

def random_sample(fq_size, sample_size):
    return random.sample(f_Q(fq_size), sample_size)

#end of copied code

A = random_sample(10000, 1000) #creates random sample

#implementation for p(k) function
def p(k,A):
    sum_length_x = 0 # initialises the sum of lengths of x (denominator)
    counter= 0 # counts how many times k appears (numorator)
    for x in A:
        cf = continued_fraction(x)
        
        sum_length_x += len(cf) -1 # -1 to remove accounting for first 0 of array
        for j in range(1,len(cf)): # starting after the 0 first index of array
            if cf[j] == k:
                counter += 1
    return counter/sum_length_x 

#the plot of p(k) against k

k_values = range(1, 20)
p_values = [p(k, A) for k in k_values]

plt.figure(figsize=(10, 6))

plt.plot(k_values, p_values, marker='o')

plt.xlabel("k values")
plt.ylabel("p(k) values")
plt.title("Plot of p(k) values against k")

plt.xticks(list(k_values))
plt.grid(True, alpha=0.3)

plt.show()